# Probe V3 Expanded E2 Mistral-7B GPU Extraction Engine

**Architecture:** `mistralai/Mistral-7B-Instruct-v0.3` ($d=4,096$, 32 Layers)
**Target Layer:** Layer 12 AST SQL Sink Pooling
**Dataset:** Expanded E2 Dataset ($N=4,673$ total: 3,304 train, 674 val, 695 test)
**Task:** Extract genuine 4096-d representations for 3,673 new samples (2,604 train, 525 val, 544 test).

> [!IMPORTANT]
> Use an environment with GPU acceleration enabled (Tesla T4, V100, A10G, or A100).


## Cell 1: Environment & GPU Hardware Verification
Confirm NVIDIA GPU and CUDA runtime are active.


In [ ]:
import sys, os, psutil, torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Model: {device_name}")
    print(f"GPU Total VRAM: {total_mem:.2f} GB")
    assert total_mem >= 14.0, "Warning: Model requires >=14GB VRAM (Tesla T4 or better) for batch_size=8."
else:
    raise RuntimeError("CUDA is not available. Please enable GPU in Runtime > Change runtime type.")


## Cell 2: Install Compatible Dependencies
Install transformers, bitsandbytes, accelerate, and dependencies.


In [ ]:
!pip install -q "transformers>=4.40.0,<4.47.0" "accelerate>=0.29.0" "bitsandbytes>=0.43.0" "safetensors" "scikit-learn"


## Cell 3: Unpack Self-Contained Extraction Payload
Unpacks scripts, AST sink extractor, and dataset split subsets.


In [ ]:
import base64, io, zipfile, os

PAYLOAD = """..."""

print("Unpacking self-contained extraction bundle...")
buf = io.BytesIO(base64.b64decode(PAYLOAD))
with zipfile.ZipFile(buf, "r") as z:
    z.extractall(".")
    print("Unpacked files:", z.namelist())


## Cell 4: Authenticate with Hugging Face (Optional/Recommended)
If needed to download mistralai/Mistral-7B-Instruct-v0.3 without rate limits.


In [ ]:
from huggingface_hub import login
import os
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in with HF_TOKEN from environment.")
else:
    print("HF_TOKEN not set; proceeding with public access / cached model.")


## Cell 5: Verify Model & AST Sink Forward Pass
Loads Mistral-7B in 4-bit NF4 and validates a test sample.


In [ ]:
!python extract_expanded_embeddings.py --mode verify --device cuda


## Cell 6: Extract Representations for New Samples
Runs extraction across the 3 unextracted sets (3,673 samples total).


In [ ]:
os.makedirs("output/new_extracted", exist_ok=True)
!python extract_expanded_embeddings.py --mode extract --data_dir data --output_dir output --batch_size 8 --device cuda


## Cell 7: Package Extracted Representations for Download
Zips the extracted .npz files into mistral_probe_v3_expanded_v2_embeddings.zip.


In [ ]:
import zipfile, glob

out_zip = "mistral_probe_v3_expanded_v2_embeddings.zip"
with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("output/new_extracted/*.npz"):
        z.write(f, os.path.basename(f))
        print("Added to zip:", f)

print(f"Successfully created {out_zip} ({os.path.getsize(out_zip) / (1024*1024):.2f} MB)!")
